# Brain Tumor MRI Classification — Colab GPU Training
**Target**: ACC ≥99.1% | AUC ≥0.995 | Sensitivity ≥96% | Specificity ≥99%

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your dataset zip to Google Drive as `/My Drive/BrainTumorAI/Project.zip`
   - The zip should contain `Project/Training/` and `Project/Testing/` folders
   - Each subfolder: `glioma/`, `meningioma/`, `notumor/`, `pituitary/`

In [ ]:
# Cell 1 — GPU check
import torch
assert torch.cuda.is_available(), "GPU not found — go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 — Install dependencies
!pip install -q timm==0.9.12 albumentations==1.3.1 torchmetrics==1.3.0
!pip install -q scikit-learn opencv-python-headless matplotlib seaborn

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# Cell 4 — Unzip dataset
import os, zipfile

ZIP_PATH  = "/content/drive/MyDrive/BrainTumorAI/Project.zip"
EXTRACT_TO = "/content/"
DATA_ROOT  = "/content/Project"

if not os.path.isdir(DATA_ROOT):
    print(f"Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_TO)
    print("Done.")
else:
    print("Dataset already extracted.")

TRAIN_DIR = os.path.join(DATA_ROOT, "Training")
TEST_DIR  = os.path.join(DATA_ROOT, "Testing")
assert os.path.isdir(TRAIN_DIR), f"Missing: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR),  f"Missing: {TEST_DIR}"
print(f"Train dir : {TRAIN_DIR}")
print(f"Test  dir : {TEST_DIR}")

In [ ]:
# Cell 5 — Imports
from __future__ import annotations
import os, random, logging, warnings, time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from scipy.special import softmax as scipy_softmax

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s — %(message)s',
                    datefmt='%H:%M:%S')
logger = logging.getLogger('BrainTumorAI')
print("Imports OK")

In [ ]:
# Cell 6 — Configuration
CFG = dict(
    # Paths
    train_dir   = "/content/Project/Training",
    test_dir    = "/content/Project/Testing",
    drive_ckpt  = "/content/drive/MyDrive/BrainTumorAI/checkpoints",
    results_dir = "/content/results",

    # Data
    class_names  = ["glioma", "meningioma", "notumor", "pituitary"],
    num_classes  = 4,
    image_size   = 224,
    val_split    = 0.20,
    seed         = 42,

    # Training
    batch_size       = 32,
    num_workers      = 2,
    phase1_epochs    = 5,
    phase2_epochs    = 20,
    patience         = 6,
    label_smoothing  = 0.1,

    # Optimizers
    lr_phase1  = 1e-3,
    lr_phase2  = 1e-4,
    weight_decay = 1e-4,

    # MixUp / CutMix
    mixup_alpha  = 0.3,
    cutmix_alpha = 0.4,

    # Ensemble weights
    ens_weights = {"efficientnet": 0.4, "resnet_cbam": 0.3, "densenet": 0.3},
)

DEVICE = torch.device("cuda")
os.makedirs(CFG["drive_ckpt"], exist_ok=True)
os.makedirs(CFG["results_dir"], exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(CFG["seed"])
print(f"Device : {DEVICE}")
print(f"Classes: {CFG['num_classes']} → {CFG['class_names']}")

In [ ]:
# Cell 7 — Build DataFrames
def build_df(root: str, class_names: List[str]) -> pd.DataFrame:
    rows = []
    for label, cname in enumerate(class_names):
        folder = Path(root) / cname
        if not folder.exists():
            # Try case-insensitive match
            matches = [p for p in Path(root).iterdir()
                       if p.name.lower() == cname.lower()]
            if matches:
                folder = matches[0]
            else:
                logger.warning(f"Folder not found: {folder}")
                continue
        for ext in ('*.jpg','*.jpeg','*.png','*.bmp'):
            for p in folder.glob(ext):
                rows.append({"filepath": str(p),
                             "label": label,
                             "class_name": cname})
    return pd.DataFrame(rows).reset_index(drop=True)


full_train_df = build_df(CFG["train_dir"], CFG["class_names"])
test_df       = build_df(CFG["test_dir"],  CFG["class_names"])

sss = StratifiedShuffleSplit(n_splits=1,
                              test_size=CFG["val_split"],
                              random_state=CFG["seed"])
train_idx, val_idx = next(sss.split(full_train_df, full_train_df["label"]))
train_df = full_train_df.iloc[train_idx].reset_index(drop=True)
val_df   = full_train_df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nTrain distribution:")
print(train_df['class_name'].value_counts().to_string())

In [ ]:
# Cell 8 — Transforms
def train_tfm(size: int) -> A.Compose:
    return A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=15, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.1, hue=0.05, p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3,5), p=1.0),
            A.GaussNoise(var_limit=(10,50), p=1.0),
        ], p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=10, p=0.4),
        A.CoarseDropout(max_holes=8, max_height=size//16,
                        max_width=size//16, p=0.3),
        A.Normalize(mean=(0.485,0.456,0.406),
                    std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


def val_tfm(size: int) -> A.Compose:
    return A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=(0.485,0.456,0.406),
                    std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


print("Transforms defined.")

In [ ]:
# Cell 9 — Dataset
class MRIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['filepath'])
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, int(row['label'])

    def sample_weights(self):
        counts = self.df['label'].value_counts()
        return (len(self.df) / self.df['label'].map(counts)).tolist()


print("MRIDataset defined.")

In [ ]:
# Cell 10 — DataLoaders
size = CFG['image_size']

train_ds = MRIDataset(train_df, train_tfm(size))
val_ds   = MRIDataset(val_df,   val_tfm(size))
test_ds  = MRIDataset(test_df,  val_tfm(size))

sw = train_ds.sample_weights()
sampler = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                          sampler=sampler,
                          num_workers=CFG['num_workers'],
                          pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'],
                          shuffle=False,
                          num_workers=CFG['num_workers'],
                          pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'],
                          shuffle=False,
                          num_workers=CFG['num_workers'],
                          pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

In [ ]:
# Cell 11 — Model Definitions

# ── EfficientNet-B3 ────────────────────────────────────────────────────────
def make_efficientnet(nc: int, pretrained: bool = True) -> nn.Module:
    m = timm.create_model('efficientnet_b3.ra2_in1k',
                          pretrained=pretrained,
                          num_classes=nc)
    return m


# ── CBAM Modules ───────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        avg = x.mean(dim=[2, 3])           # (B, C)
        mx  = x.amax(dim=[2, 3])           # (B, C)
        attn = torch.sigmoid(self.fc(avg) + self.fc(mx))
        return x * attn.view(B, C, 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg = x.mean(dim=1, keepdim=True)   # (B, 1, H, W)
        mx  = x.amax(dim=1, keepdim=True)   # (B, 1, H, W)
        attn = torch.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return x * attn


class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.sa(self.ca(x))


# ── ResNet50 + CBAM ────────────────────────────────────────────────────────
class ResNetCBAM(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = True):
        super().__init__()
        base = timm.create_model('resnet50', pretrained=pretrained,
                                 num_classes=0, global_pool='')
        self.conv1   = base.conv1
        self.bn1     = base.bn1
        self.act1    = base.act1
        self.maxpool = base.maxpool
        self.layer1  = base.layer1
        self.layer2  = base.layer2
        self.layer3  = base.layer3
        self.layer4  = base.layer4
        self.cbam3   = CBAM(1024)
        self.cbam4   = CBAM(2048)
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.head    = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act1(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.cbam3(self.layer3(x))
        x = self.cbam4(self.layer4(x))
        x = self.pool(x).flatten(1)
        return self.head(x)


def make_resnet_cbam(nc: int, pretrained: bool = True) -> nn.Module:
    return ResNetCBAM(nc, pretrained)


# ── DenseNet121 ────────────────────────────────────────────────────────────
def make_densenet(nc: int, pretrained: bool = True) -> nn.Module:
    m = timm.create_model('densenet121', pretrained=pretrained, num_classes=nc)
    return m


# ── Sanity check ───────────────────────────────────────────────────────────
nc = CFG['num_classes']
dummy = torch.randn(2, 3, 224, 224)
for name, fn in [("EfficientNet", make_efficientnet),
                 ("ResNet+CBAM",  make_resnet_cbam),
                 ("DenseNet121",  make_densenet)]:
    m = fn(nc, pretrained=False)
    out = m(dummy)
    assert out.shape == (2, nc), f"{name} output shape mismatch: {out.shape}"
    print(f"  {name:<16}: output {tuple(out.shape)} OK")
del dummy, m
print("All model architectures verified.")

In [ ]:
# Cell 12 — Training Utilities (MixUp, CutMix, EarlyStopping)

def mixup_data(x, y, alpha=0.3):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def cutmix_data(x, y, alpha=0.4):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    idx = torch.randperm(B, device=x.device)
    cut_ratio = (1 - lam) ** 0.5
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0)
    y2 = min(cy + cut_h // 2, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed_x, y, y[idx], lam


def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


class EarlyStopping:
    def __init__(self, patience: int = 6, delta: float = 1e-4):
        self.patience = patience
        self.delta    = delta
        self.best     = -float('inf')
        self.counter  = 0
        self.stop     = False

    def __call__(self, score: float) -> bool:
        if score > self.best + self.delta:
            self.best    = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


print("Utilities defined.")

In [ ]:
# Cell 13 — train_model() with AMP (NaN-safe)

def class_weights_tensor(df: pd.DataFrame, nc: int, device) -> torch.Tensor:
    counts = np.bincount(df['label'], minlength=nc).astype(float)
    w = 1.0 / (counts + 1e-8)
    w = w / w.sum() * nc
    return torch.tensor(w, dtype=torch.float32, device=device)


def train_one_epoch(model, loader, criterion, optimizer, scaler, device,
                    mixup_alpha=0.0, cutmix_alpha=0.0, augment=True):
    """
    augment=False  → plain CE loss (Phase 1, frozen backbone).
    augment=True   → MixUp / CutMix (Phase 2, full fine-tune).
    NaN / Inf loss batches are skipped safely.
    """
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    skipped = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        if augment:
            r = random.random()
            if r < 0.35 and cutmix_alpha > 0:
                imgs, ya, yb, lam = cutmix_data(imgs, labels, cutmix_alpha)
            elif r < 0.65 and mixup_alpha > 0:
                imgs, ya, yb, lam = mixup_data(imgs, labels, mixup_alpha)
            else:
                ya, yb, lam = labels, labels, 1.0
        else:
            ya, yb, lam = labels, labels, 1.0

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(imgs)
            loss   = mixup_loss(criterion, logits, ya, yb, lam)

        # Skip batch if loss is NaN or Inf (AMP overflow protection)
        if not torch.isfinite(loss):
            skipped += 1
            scaler.update()          # keep scaler state consistent
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        with torch.no_grad():
            preds = logits.detach().float().argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)

    if skipped:
        print(f"    [warn] skipped {skipped} NaN/Inf batches")

    if total == 0:
        return float('nan'), 0.0
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)

        if not torch.isfinite(loss):
            continue

        total_loss += loss.item() * imgs.size(0)
        # float32 cast prevents float16 row-sum != 1 errors
        logits_f32 = torch.nan_to_num(logits.float(), nan=0.0, posinf=0.0, neginf=0.0)
        probs  = F.softmax(logits_f32, dim=1)
        preds  = probs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    if not all_probs:
        return float('nan'), 0.0, 0.0

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    # Guard: replace any residual NaN rows with uniform distribution
    nan_rows = ~np.isfinite(all_probs).all(axis=1)
    if nan_rows.any():
        all_probs[nan_rows] = 1.0 / all_probs.shape[1]
        print(f"    [warn] {nan_rows.sum()} NaN prob rows replaced with uniform")

    # Re-normalise to exactly 1.0
    row_sums = all_probs.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    all_probs = all_probs / row_sums

    try:
        auc = roc_auc_score(all_labels, all_probs,
                            multi_class='ovr', average='macro')
    except ValueError as e:
        print(f"    [warn] roc_auc_score failed: {e} — returning 0.0")
        auc = 0.0

    return total_loss / max(total, 1), correct / max(total, 1), auc


def freeze_backbone(model):
    for name, p in model.named_parameters():
        if any(k in name for k in ('classifier','head','fc','cbam')):
            p.requires_grad = True
        else:
            p.requires_grad = False


def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True


def train_model(model, model_name: str, train_loader, val_loader,
                train_df, device, cfg: dict):

    ckpt_dir = Path(cfg['drive_ckpt']) / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    nc = cfg['num_classes']
    cw = class_weights_tensor(train_df, nc, device)
    criterion = nn.CrossEntropyLoss(weight=cw,
                                    label_smoothing=cfg['label_smoothing'])

    # init_scale=256 avoids early AMP overflow that causes NaN weights
    scaler   = GradScaler(init_scale=256)
    best_auc = 0.0
    best_path: Optional[Path] = None

    print(f"\n{'#'*55}")
    print(f"  Training: {model_name.upper()}")
    print(f"{'#'*55}")

    # ── Phase 1: Head warm-up (no augment — frozen backbone is unstable with MixUp) ──
    freeze_backbone(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Phase 1 — trainable params: {trainable:,}")

    opt1 = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg['lr_phase1']
    )
    scheduler1 = torch.optim.lr_scheduler.OneCycleLR(
        opt1, max_lr=cfg['lr_phase1'],
        steps_per_epoch=len(train_loader),
        epochs=cfg['phase1_epochs'],
        pct_start=0.3, anneal_strategy='cos'
    )

    for ep in range(1, cfg['phase1_epochs'] + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, opt1, scaler, device,
            augment=False)                  # no MixUp/CutMix in Phase 1
        scheduler1.step()
        vl_loss, vl_acc, vl_auc = validate(model, val_loader, criterion, device)
        elapsed = time.time() - t0

        if np.isfinite(vl_auc) and vl_auc > best_auc:
            best_auc  = vl_auc
            best_path = ckpt_dir / f"best_p1_ep{ep:02d}_auc{vl_auc:.4f}.pt"
            torch.save({'model_state_dict': model.state_dict(),
                        'auc': vl_auc, 'acc': vl_acc,
                        'epoch': ep, 'phase': 1}, best_path)
            print(f"    Saved: auc={vl_auc:.4f} acc={vl_acc:.4f}")

        print(f"  [P1 {ep:02d}/{cfg['phase1_epochs']}] "
              f"tr_loss={tr_loss:.4f} val_loss={vl_loss:.4f} "
              f"val_acc={vl_acc:.4f} val_auc={vl_auc:.4f} ({elapsed:.0f}s)")

    # ── Phase 2: Full fine-tune ────────────────────────────────────────────
    if best_path and best_path.exists():
        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"  Loaded Phase 1 best (auc={best_auc:.4f}) for Phase 2")

    unfreeze_all(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  Phase 2 — trainable params: {trainable:,}")

    opt2 = torch.optim.AdamW(model.parameters(),
                              lr=cfg['lr_phase2'],
                              weight_decay=cfg['weight_decay'])
    scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=cfg['phase2_epochs'], eta_min=1e-6)

    # Reset scaler for Phase 2
    scaler2 = GradScaler(init_scale=2**16)
    es = EarlyStopping(patience=cfg['patience'])

    for ep in range(1, cfg['phase2_epochs'] + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, opt2, scaler2, device,
            mixup_alpha=cfg['mixup_alpha'],
            cutmix_alpha=cfg['cutmix_alpha'],
            augment=True)
        vl_loss, vl_acc, vl_auc = validate(model, val_loader, criterion, device)
        scheduler2.step()
        elapsed = time.time() - t0

        if np.isfinite(vl_auc) and vl_auc > best_auc:
            best_auc  = vl_auc
            best_path = ckpt_dir / f"best_p2_ep{ep:02d}_auc{vl_auc:.4f}.pt"
            torch.save({'model_state_dict': model.state_dict(),
                        'auc': vl_auc, 'acc': vl_acc,
                        'epoch': ep, 'phase': 2}, best_path)
            print(f"    Saved: auc={vl_auc:.4f} acc={vl_acc:.4f}")

        print(f"  [P2 {ep:02d}/{cfg['phase2_epochs']}] "
              f"tr_loss={tr_loss:.4f} val_loss={vl_loss:.4f} "
              f"val_acc={vl_acc:.4f} val_auc={vl_auc:.4f} ({elapsed:.0f}s)")

        if np.isfinite(vl_auc) and es(vl_auc):
            print(f"  Early stopping at epoch {ep} (patience={cfg['patience']})")
            break

    if best_path and best_path.exists():
        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"  Final best: {best_path.name}  (auc={best_auc:.4f})")

    return model, best_auc


print("train_model() defined.")


In [ ]:
# Cell 14 — Ensemble + Temperature Calibration

class Ensemble(nn.Module):
    """Weighted soft-voting ensemble with per-model temperature scaling."""

    def __init__(self, models: Dict[str, nn.Module], weights: Dict[str, float]):
        super().__init__()
        self.models = nn.ModuleDict(models)
        self.weights = weights
        # Learnable temperature per model (initialized to 1.0)
        self.temps = nn.ParameterDict({
            k: nn.Parameter(torch.ones(1)) for k in models
        })

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        total = None
        for k, m in self.models.items():
            logits = m(x) / self.temps[k].clamp(min=0.1)
            prob   = F.softmax(logits, dim=1)
            w      = self.weights.get(k, 1.0)
            total  = w * prob if total is None else total + w * prob
        return total  # already normalized if weights sum to 1


def calibrate_temperature(ensemble: Ensemble, val_loader, device, lr=0.01, n_iter=100):
    """LBFGS temperature scaling calibration on validation set."""
    ensemble.eval()
    # Freeze all model params, only optimize temperatures
    for p in ensemble.models.parameters():
        p.requires_grad_(False)
    for p in ensemble.temps.parameters():
        p.requires_grad_(True)

    # Collect logits and labels
    all_logits: Dict[str, List] = {k: [] for k in ensemble.models}
    all_labels = []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            for k, m in ensemble.models.items():
                all_logits[k].append(m(imgs).cpu())
            all_labels.append(labels)

    stacked = {k: torch.cat(v).to(device) for k, v in all_logits.items()}
    labels_t = torch.cat(all_labels).to(device)

    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.LBFGS(list(ensemble.temps.parameters()),
                             lr=lr, max_iter=n_iter)

    def closure():
        opt.zero_grad()
        total = None
        for k in ensemble.models:
            logits = stacked[k] / ensemble.temps[k].clamp(min=0.1)
            prob   = F.softmax(logits, dim=1)
            w      = ensemble.weights.get(k, 1.0)
            total  = w * prob if total is None else total + w * prob
        loss = criterion(torch.log(total + 1e-8), labels_t)
        loss.backward()
        return loss

    opt.step(closure)

    for k in ensemble.models:
        t = ensemble.temps[k].item()
        print(f"  Temperature [{k}]: {t:.4f}")

    # Re-enable model grads for future eval
    for p in ensemble.models.parameters():
        p.requires_grad_(False)
    for p in ensemble.temps.parameters():
        p.requires_grad_(False)

    return ensemble


print("Ensemble & calibration defined.")

In [ ]:
# Cell 15 — Evaluate + Bootstrap CI

def compute_specificity(y_true: np.ndarray, y_pred: np.ndarray,
                         num_classes: int) -> np.ndarray:
    specs = []
    for c in range(num_classes):
        tn = ((y_true != c) & (y_pred != c)).sum()
        fp = ((y_true != c) & (y_pred == c)).sum()
        specs.append(tn / (tn + fp + 1e-8))
    return np.array(specs)


def compute_sensitivity(y_true: np.ndarray, y_pred: np.ndarray,
                         num_classes: int) -> np.ndarray:
    senses = []
    for c in range(num_classes):
        tp = ((y_true == c) & (y_pred == c)).sum()
        fn = ((y_true == c) & (y_pred != c)).sum()
        senses.append(tp / (tp + fn + 1e-8))
    return np.array(senses)


def safe_probs(out_tensor: torch.Tensor) -> np.ndarray:
    """Convert model output to clean float32 probability array."""
    out_f32 = torch.nan_to_num(out_tensor.float(), nan=0.0, posinf=0.0, neginf=0.0)
    row_sum  = out_f32.sum(dim=1, keepdim=True)
    # Ensemble already returns probs (sum ~ 1); single models return logits
    is_prob = (row_sum - 1.0).abs().mean().item() < 0.05
    if is_prob:
        probs = (out_f32 / row_sum.clamp(min=1e-8))
    else:
        probs = F.softmax(out_f32, dim=1)
    arr = probs.cpu().numpy()
    # Replace any remaining NaN rows with uniform
    nan_rows = ~np.isfinite(arr).all(axis=1)
    if nan_rows.any():
        arr[nan_rows] = 1.0 / arr.shape[1]
    # Re-normalise
    rs = arr.sum(axis=1, keepdims=True)
    rs = np.where(rs == 0, 1.0, rs)
    return arr / rs


@torch.no_grad()
def evaluate(model, loader, device, num_classes: int,
             class_names: List[str], label: str = '') -> dict:
    model.eval()
    all_probs, all_labels = [], []

    for imgs, labels in loader:
        imgs = imgs.to(device)
        with autocast():
            out = model(imgs)
        all_probs.append(safe_probs(out))
        all_labels.append(labels.numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)
    y_pred = y_prob.argmax(axis=1)

    acc  = accuracy_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    sens = compute_sensitivity(y_true, y_pred, num_classes).mean()
    spec = compute_specificity(y_true, y_pred, num_classes).mean()
    cm   = confusion_matrix(y_true, y_pred)

    tag = f' [{label}]' if label else ''
    print(f"\n{'='*55}")
    print(f"  Evaluation{tag}")
    print(f"{'='*55}")
    print(f"  Accuracy    : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  AUC (macro) : {auc:.4f}")
    print(f"  Sensitivity : {sens:.4f}")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1 (macro)  : {f1:.4f}")
    print(f"{'='*55}")

    return dict(y_true=y_true, y_pred=y_pred, y_prob=y_prob,
                accuracy=acc, auc=auc, sensitivity=sens,
                specificity=spec, f1_macro=f1, cm=cm)


def bootstrap_ci(y_true, y_pred, y_prob,
                  n_iterations=1000, confidence=0.95,
                  num_classes=4, random_state=42) -> dict:
    alpha   = 1.0 - confidence
    rng     = np.random.RandomState(random_state)
    n       = len(y_true)
    metrics = {'accuracy':[], 'auc':[], 'sensitivity':[], 'specificity':[], 'f1_macro':[]}

    for _ in range(n_iterations):
        idx = rng.randint(0, n, n)
        bt, bp, bpr = y_true[idx], y_pred[idx], y_prob[idx]
        if len(np.unique(bt)) < 2:
            continue
        # renorm bootstrap probs
        bpr = bpr / bpr.sum(axis=1, keepdims=True).clip(1e-8)
        metrics['accuracy'].append(accuracy_score(bt, bp))
        metrics['f1_macro'].append(f1_score(bt, bp, average='macro', zero_division=0))
        metrics['sensitivity'].append(compute_sensitivity(bt, bp, num_classes).mean())
        metrics['specificity'].append(compute_specificity(bt, bp, num_classes).mean())
        try:
            metrics['auc'].append(
                roc_auc_score(bt, bpr, multi_class='ovr', average='macro'))
        except ValueError:
            pass

    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    results = {}
    for k, v in metrics.items():
        if not v:
            results[k] = dict(mean=float('nan'), lower=float('nan'),
                              upper=float('nan'), std=float('nan'))
        else:
            a = np.array(v)
            results[k] = dict(mean=a.mean(), lower=np.percentile(a, lo),
                              upper=np.percentile(a, hi), std=a.std())
    return results


def format_ci_table(results: dict) -> str:
    lines = ['| Metric | Mean | 95% CI | Std |', '|---|---|---|---|']
    for metric, v in results.items():
        ci = f"[{v['lower']:.4f}, {v['upper']:.4f}]"
        lines.append(f"| {metric} | {v['mean']:.4f} | {ci} | {v['std']:.4f} |")
    return '\n'.join(lines)


print("Evaluation functions defined.")


In [ ]:
# Cell 16 — Train all 3 models sequentially
# Expected time on T4 GPU: ~40-80 min total

nc     = CFG['num_classes']
models_trained: Dict[str, nn.Module] = {}
val_aucs: Dict[str, float] = {}

model_fns = [
    ('efficientnet', make_efficientnet),
    ('resnet_cbam',  make_resnet_cbam),
    ('densenet',     make_densenet),
]

for mname, mfn in model_fns:
    torch.cuda.empty_cache()
    set_seed(CFG['seed'])                      # reproducibility per model
    model = mfn(nc, pretrained=True).to(DEVICE)
    model, best_auc = train_model(
        model, mname, train_loader, val_loader,
        train_df, DEVICE, CFG
    )
    models_trained[mname] = model
    val_aucs[mname]       = best_auc
    torch.cuda.empty_cache()

print("\n" + "="*55)
print("  All models trained:")
for k, v in val_aucs.items():
    status = "OK" if v >= 0.99 else ("GOOD" if v >= 0.97 else "CHECK")
    print(f"    [{status}] {k:<16}: val_auc={v:.4f}")
print("="*55)


In [ ]:
# Cell 17 — Build Ensemble & Calibrate

ensemble = Ensemble(models_trained, CFG['ens_weights']).to(DEVICE)
ensemble.eval()

print("Calibrating ensemble temperatures on validation set...")
ensemble = calibrate_temperature(ensemble, val_loader, DEVICE)

# Save calibrated ensemble
ens_path = Path(CFG['drive_ckpt']) / 'ensemble_final.pt'
torch.save(ensemble.state_dict(), ens_path)
print(f"Saved calibrated ensemble to: {ens_path}")

In [ ]:
# Cell 18 — Validation evaluation (all models + ensemble)

val_results = {}
print("\n" + "="*55)
print("  VALIDATION SET EVALUATION")
print("="*55)

for mname, m in models_trained.items():
    res = evaluate(m, val_loader, DEVICE, nc, CFG['class_names'],
                   label=f"{mname} (VAL)")
    val_results[mname] = res

ens_val = evaluate(ensemble, val_loader, DEVICE, nc,
                   CFG['class_names'], label="ENSEMBLE (VAL)")
val_results['ensemble'] = ens_val

# Bootstrap CI on ensemble val
print("\nComputing 95% Bootstrap CI on VAL (n=1000)...")
ci_val = bootstrap_ci(ens_val['y_true'], ens_val['y_pred'], ens_val['y_prob'],
                       n_iterations=1000, num_classes=nc)
from IPython.display import Markdown, display
display(Markdown(format_ci_table(ci_val)))

In [ ]:
# Cell 19 — Final locked test set evaluation

print("\n" + "="*55)
print("  FINAL TEST SET EVALUATION")
print("="*55)

test_rows = []
for mname, m in models_trained.items():
    res = evaluate(m, test_loader, DEVICE, nc,
                   CFG['class_names'], label=f"{mname} (TEST)")
    test_rows.append({'model': mname, 'split': 'test',
                      'accuracy': res['accuracy'], 'auc': res['auc'],
                      'sensitivity': res['sensitivity'],
                      'specificity': res['specificity'],
                      'f1_macro': res['f1_macro']})

ens_test = evaluate(ensemble, test_loader, DEVICE, nc,
                    CFG['class_names'], label="ENSEMBLE (TEST)")
test_rows.append({'model': 'ensemble', 'split': 'test',
                  'accuracy': ens_test['accuracy'], 'auc': ens_test['auc'],
                  'sensitivity': ens_test['sensitivity'],
                  'specificity': ens_test['specificity'],
                  'f1_macro': ens_test['f1_macro']})

# Bootstrap CI on test
print("\nComputing 95% Bootstrap CI on TEST (n=1000)...")
ci_test = bootstrap_ci(ens_test['y_true'], ens_test['y_pred'], ens_test['y_prob'],
                        n_iterations=1000, num_classes=nc)
display(Markdown(format_ci_table(ci_test)))

# Compare to base paper
base = {'accuracy': 0.971, 'auc': 0.980, 'sensitivity': 0.919, 'specificity': 0.980}
ens = test_rows[-1]
print(f"\n{'='*55}")
print(f"  ENSEMBLE vs. Amin et al. 2020 (SVM+GLCM)")
print(f"{'='*55}")
for m, bv in base.items():
    ov = ens.get(m, 0)
    status = '✓' if ov >= bv else '✗'
    print(f"  {status} {m:<15}: {bv:.3f} → {ov:.4f}  Δ={ov-bv:+.4f}")
print(f"{'='*55}")

In [ ]:
# Cell 20 — Confusion matrix + per-class metrics plots

def plot_confusion_matrix(cm: np.ndarray, class_names: List[str],
                          title: str = 'Confusion Matrix',
                          save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


def plot_roc_curves(y_true: np.ndarray, y_prob: np.ndarray,
                    class_names: List[str],
                    save_path: Optional[str] = None):
    from sklearn.metrics import roc_curve, auc
    from sklearn.preprocessing import label_binarize

    nc = len(class_names)
    y_bin = label_binarize(y_true, classes=list(range(nc)))

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = plt.cm.tab10(np.linspace(0, 0.9, nc))
    for i, (cname, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f'{cname} (AUC={roc_auc:.4f})')

    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate',  fontsize=12)
    ax.set_title('ROC Curves (Ensemble — Test Set)', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()


results_dir = Path(CFG['results_dir'])
results_dir.mkdir(exist_ok=True)

plot_confusion_matrix(
    ens_test['cm'], CFG['class_names'],
    title='Ensemble — Test Set Confusion Matrix',
    save_path=str(results_dir / 'cm_ensemble_test.png')
)

plot_roc_curves(
    ens_test['y_true'], ens_test['y_prob'],
    CFG['class_names'],
    save_path=str(results_dir / 'roc_ensemble_test.png')
)

In [ ]:
# Cell 21 — Save everything to Google Drive

drive_results = Path('/content/drive/MyDrive/BrainTumorAI/results')
drive_results.mkdir(parents=True, exist_ok=True)

# Save metrics CSV
metrics_df = pd.DataFrame(test_rows)
csv_path   = results_dir / 'final_test_metrics.csv'
metrics_df.to_csv(csv_path, index=False)
print(f"Saved metrics CSV: {csv_path}")

# Copy plots and CSV to Drive
import shutil
for fname in ['cm_ensemble_test.png', 'roc_ensemble_test.png', 'final_test_metrics.csv']:
    src = results_dir / fname
    if src.exists():
        shutil.copy(src, drive_results / fname)
        print(f"Copied to Drive: {fname}")

# Copy checkpoints to Drive (already saved there)
print(f"\nCheckpoints saved to: {CFG['drive_ckpt']}")
print("\nAll done! Your brain tumor AI results are saved to Google Drive.")

# Final summary
ens = test_rows[-1]
targets = {'accuracy': 0.991, 'auc': 0.995, 'sensitivity': 0.96, 'specificity': 0.99}
print(f"\n{'='*55}")
print(f"  TARGET vs. ACHIEVED (Ensemble — Test Set)")
print(f"{'='*55}")
for m, target in targets.items():
    achieved = ens.get(m, 0)
    status = '✓ PASS' if achieved >= target else '✗ MISS'
    print(f"  {status}  {m:<15}: target={target:.3f}  got={achieved:.4f}")
print(f"{'='*55}")